In [1]:
import pickle
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm
import time, yaml, pickle
from datetime import datetime
from os.path import join
import multiprocessing
from functools import partial
from multiprocessing import Pool
from itertools import product

tqdm.pandas()

### Begin home location to MRMS matching

#### Load home location file

In [10]:
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")
for date in AR_dates:
    print(files[0].replace("2023-03-15", date.strftime('%Y-%m-%d')))

/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2024-03-29_PST_chunked.parquet
/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2024-03-30_PST_chunked.parquet
/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2024-03-31_PST_chunked.parquet
/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2024-04-01_PST_chunked.parquet


In [5]:
import os
base_data_path = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/"
start_date = "2023-03-15"
end_date = "2024-04-29"

date_range = pd.date_range(start=start_date, end=end_date, freq='d')
data_files = date_range.strftime(f"{base_data_path}veraset_%Y-%m-%d_PST_chunked.parquet")
files = []
for i, f in enumerate(data_files):
    if os.path.isfile(f):
        files.append(f)
print(len(files))

92


In [11]:
df_temp = pd.read_parquet(files[0])
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
218,0000ba15f25930f0a4ad2bd7114a40bd30a14214b245a8...,3 days 13:28:52,33.929420,-118.051904,90605,0,LA,88,10.133705,"[0.0007038977772365412, 0.0, 0.000821825114659...",1.333580,10,3.259706,2023-03-15,False,7.3,46.9,1.9,20.0
248,0000a843eb75661d74cd9ba58f6830a1f84c5f49a0eac3...,0 days 00:23:52,33.927723,-118.203461,90262,0,LA,7,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.001408,1,0.000000,2023-03-15,False,7.9,51.9,1.0,13.1
260,0000a396879fd5767b3d146bfe8c854ea48c8e9a86b9f5...,4 days 18:17:15,34.162756,-119.168017,93033,0,VE,81,23.544740,"[0.22042789638748198, 0.34614753068159015, 0.0...",1.880407,5,6.822220,2023-03-15,False,7.5,35.2,1.2,13.5
475,0000712fa93eae897b02b840ce7a3be2b9eb8b06b12c46...,3 days 09:06:21,34.087652,-118.287289,90029,0,LA,15,1.356103,"[0.0022030198624645985, 0.0022030198624645985,...",0.158470,3,0.630850,2023-03-15,False,9.0,51.1,1.0,9.7
485,00006558db84477cda9278ba16d40cab37598b931ad2e1...,1 days 22:01:25,34.157461,-118.006886,91016,0,LA,2,0.001101,[0.001101141416629911],0.000653,1,0.001115,2023-03-15,False,11.6,70.7,1.9,16.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,fffff922ba116e91a1274046b2f7b158355940c5433bb7...,0 days 00:27:11,33.742100,-118.310204,90732,75017,LA,4,0.000000,"[0.0, 0.0, 0.0]",0.000000,1,0.000000,2023-03-15,False,7.3,56.1,2.5,18.1
37,ffffe79bdb634159d1e5d074c662e2336fe012548ed571...,1 days 22:53:37,34.394936,-118.908152,93015,75017,VE,16,33.708051,"[1.1559947293736152, 1.29116154228278, 1.14343...",3.359933,4,15.526305,2023-03-15,False,8.0,43.9,1.7,18.0
280,ffffc189a959d16f168dfb7f90ebf0355d8f1cfae1e797...,3 days 22:55:35,33.889900,-118.400803,90266,75017,LA,243,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.001471,1,0.000000,2023-03-15,False,7.7,45.8,0.8,6.5
336,ffffacf186455724eff8a4c113f92378a24f2a6fa25d5d...,5 days 09:35:59,34.277335,-118.405703,91331,75017,LA,336,8.636476,"[0.0012725249472048672, 0.019297951426232113, ...",0.672377,1,3.176273,2023-03-15,False,7.1,40.6,2.5,19.3


In [16]:
df_temp["FFW"]

218    False
248    False
260    False
475    False
485    False
       ...  
10     False
37     False
280    False
336    False
438    False
Name: FFW, Length: 681854, dtype: bool

In [15]:
df_temp["FFW"].rename(f"FFW_{date.strftime('%Y-%m-%d')}")

218    False
248    False
260    False
475    False
485    False
       ...  
10     False
37     False
280    False
336    False
438    False
Name: FFW_2024-04-01, Length: 681854, dtype: bool

In [6]:
files[0]

'/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2023-03-15_PST_chunked.parquet'

In [8]:
files[0].replace("2023-03-15", date.strftime('%Y-%m-%d'))

'/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251116/veraset_2024-04-01_PST_chunked.parquet'

In [2]:
home_loc_file = "/glade/work/ggantos/cat-rain/notebooks/home_loc_by_row_group.parquet"
home_loc_df = pd.read_parquet(home_loc_file)
print(f"Original home locations: {home_loc_df.shape[0]:,}")

with open("../config/zips_counties.yaml", 'r') as stream:
    zips_counties = yaml.load(stream, Loader=yaml.FullLoader)
    zips = [str(z) for z in zips_counties["zips_LA"] + zips_counties["zips_SB"] + zips_counties["zips_VE"]]
zips_dict = {}
for k,v in zips_counties.items():
    for x in v:
        zips_dict[str(x)] = k.split("_")[1]

home_loc_df["county"] = home_loc_df["zipcode"].map(zips_dict)
df_gb = home_loc_df.groupby('county').size()
print(f"By County:\n\t{df_gb.index[0]}: {df_gb.iloc[0]:,}\n\t{df_gb.index[1]}: {df_gb.iloc[1]:,}\n\t{df_gb.index[2]}: {df_gb.iloc[2]:,}")

Original home locations: 3,385
By County:
	LA: 2,926
	SB: 145
	VE: 314


### Match home locations to nearest MRMS data

In [3]:
def create_tree(mrms_ds):
    # define projection
    proj = Proj(proj="lcc", R=6371229, lat_0=25, lon_0=265, lat_1=25, lat_2=25)
    
    # grid and project MRMS lat-lon coords then create KDTree
    mrms_lon_grid = np.tile(mrms_ds['lon'].values-360, (mrms_ds['lat'].values.shape[0], 1)).T
    mrms_lat_grid = np.tile(mrms_ds['lat'].values, (mrms_ds['lon'].values.shape[0], 1))
    proj_mrms_lon_grid, proj_mrms_lat_grid = np.array(proj(mrms_lon_grid, mrms_lat_grid))
    tree = cKDTree(np.c_[proj_mrms_lon_grid.ravel(), proj_mrms_lat_grid.ravel()])
    return tree, proj, mrms_lon_grid.shape

def get_mrms_var(df_sub, idx, mrms_ds, variable, variable_name):

    # grab MRMS data using calculated index
    lon_idx_da = xr.DataArray(idx[:, 0], dims="lon")
    lat_idx_da = xr.DataArray(idx[:, 1], dims="lat")
    mrms_idx = mrms_ds.isel(lon=lon_idx_da).isel(lat=lat_idx_da)[variable].values
    mrms_idx = np.swapaxes(mrms_idx, 0, 2)
    mrms_idx = mrms_idx[np.arange(0, mrms_idx.shape[0]), np.arange(0, mrms_idx.shape[1]), :]
    series = pd.Series(mrms_idx.tolist(), name=variable_name, index=df_sub.index)
    return series

def cycle_through_mrms_variables(row_group, home_loc_df, variables, date_strs, variable_aggregates):

    mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variables[0]}_{date_strs[0]}_daily_{variable_aggregates[0]}.nc"
    mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
    tree, proj, mrms_lon_grid_shape = create_tree(mrms_ds)
    
    df_sub = home_loc_df[home_loc_df.row_group == row_group]
    _, points = tree.query(np.column_stack(proj(df_sub['lng'], df_sub['lat'])))
    idx = np.column_stack(np.unravel_index(points, mrms_lon_grid_shape))

    for variable, date_str, variable_aggregate in product(variables, date_strs, variable_aggregates):
        mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variable}_{date_str}_daily_{variable_aggregate}.nc"
        mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
        variable_name = f"streamflow_{variable_aggregate}" if "STREAMFLOW" in variable else f"precip_{variable_aggregate}"
        series = get_mrms_var(df_sub, idx, mrms_ds, variable, variable_name)
        if variable_name in df_sub.columns:
            df_sub[variable_name] = df_sub[variable_name] + series
        else:
            df_sub = df_sub.merge(series, left_index=True, right_index=True)
    return df_sub

In [4]:
date_strs = ["2024-03-15T00:00_2024-04-29T00:00", "2023-03-15T00:00_2023-04-29T00:00"]
variables = ["MultiSensor_QPE_01H_Pass2_00.00", "FLASH_HP_MAXUNITSTREAMFLOW_00.00"]
variable_aggregates = ["max", "sum"]
row_groups = home_loc_df.row_group.unique()

n_processors = multiprocessing.cpu_count()
f = partial(cycle_through_mrms_variables,
            home_loc_df=home_loc_df,
            variables=variables,
            date_strs=date_strs,
            variable_aggregates=variable_aggregates)

s = time.time()
print('Beginning data processing...')
with Pool(processes=n_processors) as pool:
    result = pool.map_async(f, row_groups)
    results = result.get()
print(f"{np.round((time.time() - s) / 60, 2)} minutes to complete parallel computation.")
combined_data = pd.concat(results)
combined_data

Beginning data processing...


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return self._transformer._transform_point(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/pyproj/transformer.py:817: FutureWarning: Calling float on a single element Series is deprecated an

2.51 minutes to complete parallel computation.


,duration,lat,lng,zipcode,row_group,county,precip_max,precip_sum,streamflow_max,streamflow_sum
uid,,,,,,,,,,
001863256a1b0ecec4d2acff443b07351c8331b278c9d43e61ca7939753c74d3,26 days 12:07:27,34.153925,-118.227728,91206,28,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.7, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.2, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.2, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5, ..."
00274bef19119820a22f4ab3c300ba3fc81e9156a07b0fe81e081bded2f64f24,24 days 16:14:35,33.972454,-118.285176,90044,45,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.300...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.400...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
004988492810559122f298d8da64b152d0545cf4195475358234eef096b05ab4,41 days 01:21:55,34.192537,-118.429267,91401,84,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.300...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.2,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3, ..."
0057b75913984f1ce0eedd7e7f3be6e9cabba56f75c73a87d908c60d5867bedf,50 days 14:45:28,34.102497,-117.776870,91750,100,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ..."
0090c9be63a8530db4c81557aba47cb4ad07a96861ff4df6925303af625de55e,30 days 03:52:51,34.069772,-118.307110,90004,165,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.900...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.300...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.300..."
...,...,...,...,...,...,...,...,...,...,...
ffa764a613481d6800a22e2237b01c056917e9bb69c282a09c2f60b0a29c83fc,21 days 01:21:21,34.095121,-118.074165,91776,74916,LA,"[0.0, 16.8, 2.2, 0.0, 0.0, 0.0, 0.0, 0.0, 1.20...","[0.0, 19.8, 2.2, 0.0, 0.0, 0.0, 0.0, 0.0, 3.90...","[0.0, 1.3, 0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, ...","[0.0, 2.5000000000000004, 1.0999999999999999, ..."
ffb741b50f69eae60289ee5719a32e9274b4ca255ee6e8bfe514360f72b21798,27 days 02:57:31,34.181711,-118.617836,91367,74934,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.3,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.999..."
ffe3db49bc6a1b2a7eaea60138e3b1e8ed48d64399d93d8415420058afc9b906,41 days 23:57:27,34.054257,-117.759149,91766,74985,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.700...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.200..."


In [5]:
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}.parquet"
combined_data.to_parquet(join(save_out_path, file_path))

### Calculate Mobility Metrics Baseline Statistics

In [6]:
all_dates = pd.date_range(date_strs[0].split("_")[0], date_strs[0].split("_")[1], freq="1d").union(pd.date_range(date_strs[1].split("_")[0], date_strs[1].split("_")[1], freq="1d"))
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")
baseline_dates = all_dates.difference(AR_dates)

In [29]:
column_heads=['base_all', 'base_weekend', 'base_weekday', 'base_fri', 'base_sat', 'base_sun', 'base_mon']
column_subs = ['mean', 'std', 'median', 'iqr', 'n']
columns = [c+'_'+cs for c in column_heads for cs in column_subs]
mobility_metrics = ['distance_straight_line', 'max_distance_from_home', 'num_stops']
columns = [mm+'_'+c for c in columns for mm in mobility_metrics]
column_heads_weekdays = {'base_all': [0,1,2,3,4,5,6],
                         'base_weekend': [5,6],
                         'base_weekday': [0,1,2,3,4],
                         'base_fri': [4],
                         'base_sat': [5],
                         'base_sun': [6],
                         'base_mon': [0]}

baseline_uids = {'base_all': 'uids_20251029_1740.parquet',
                 'base_weekend': 'uids_20251029_0343.parquet',
                 'base_weekday': 'uids_20251103_1036.parquet',
                 'base_fri': 'uids_20251028_1236.parquet',
                 'base_sat': 'uids_20251028_1253.parquet',
                 'base_sun': 'uids_20251028_1307.parquet',
                 'base_mon': 'uids_20251028_1312.parquet'}

df = pd.DataFrame(index=combined_data.index, columns=columns)

series_list = []
for baseline_date in baseline_dates:
    weekday = baseline_date.day_of_week
    data_dir = f"/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251110/"
    fn = f"veraset_{baseline_date.strftime('%Y-%m-%d')}_PST_chunked.parquet"
    metrics = pd.read_parquet(data_dir + fn).set_index("uid", drop=True)
    # print(f"Overlap UIDs: {np.intersect1d(metrics.uid.unique(), combined_data.index.unique()).shape}")
    # metrics = metrics[metrics.uid.isin(combined_data.index)].set_index("uid", drop=True)

    for column_head in column_heads:
        with open(f"/glade/campaign/cisl/aiml/cat_flood/uids/{baseline_uids[column_head]}", 'rb') as file:
            baseline_uids_list = pickle.load(file)
        metrics_baseline = metrics[metrics.index.isin(baseline_uids_list)]
        if weekday in column_heads_weekdays[column_head]:
            for mobility_metric in mobility_metrics:
                col = mobility_metric + '_' + column_head
                series_list.append(pd.Series(metrics_baseline[mobility_metric], name=col, index=metrics_baseline.index))
        else:
            continue

In [33]:
def calculate_iqr(row):
    Q1 = row.quantile(0.25)
    Q3 = row.quantile(0.75)
    IQR = Q3 - Q1
    return IQR

series = pd.concat(series_list, axis=1, join="inner")
df = df[df.index.isin(series.index)]
for column_head in column_heads:
    for mobility_metric in mobility_metrics:
        col = mobility_metric + "_" + column_head
        df.loc[:, col + "_mean"] = series[col].mean(axis=1)
        df.loc[:, col + "_std"] = series[col].std(axis=1)
        df.loc[:, col + "_median"] = series[col].median(axis=1)
        df.loc[:, col + "_iqr"] = series[col].apply(calculate_iqr, axis=1)
        df.loc[:, col + "_n"] = series[col].count(axis=1)
df

,distance_straight_line_base_all_mean,max_distance_from_home_base_all_mean,num_stops_base_all_mean,distance_straight_line_base_all_std,max_distance_from_home_base_all_std,num_stops_base_all_std,distance_straight_line_base_all_median,max_distance_from_home_base_all_median,num_stops_base_all_median,distance_straight_line_base_all_iqr,...,num_stops_base_mon_std,distance_straight_line_base_mon_median,max_distance_from_home_base_mon_median,num_stops_base_mon_median,distance_straight_line_base_mon_iqr,max_distance_from_home_base_mon_iqr,num_stops_base_mon_iqr,distance_straight_line_base_mon_n,max_distance_from_home_base_mon_n,num_stops_base_mon_n
uid,,,,,,,,,,,,,,,,,,,,,
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,72.203654,17.358913,5.0,50.767251,13.044697,2.165561,55.30733,14.354902,5.0,35.398744,...,2.605356,52.195381,14.03958,4.0,30.059935,1.786403,4.0,12,12,12
014d302436233505e32eb98250e9fae52e0cd079269ed8e3949838f740aeedd3,70.34959,8.421426,8.545455,118.50602,11.899245,3.132731,25.833019,4.911474,8.0,58.033164,...,3.895413,31.020494,6.044708,11.0,54.743476,4.995454,4.5,12,12,12
0263a0771738b6f9c0e2bf6ab61b963da028a0ef0dfa6ffb7db574441a66789c,411.656516,136.790508,6.534091,673.466929,168.593437,3.205415,120.87848,32.615257,7.0,358.035184,...,2.269695,103.989883,21.725303,6.5,89.869062,106.681447,3.0,12,12,12
0289ccadab6bfc8e61b02a43b73a75c29af9e720b8b5acdd245d89dfccb5bf46,208.81581,50.450195,3.488636,653.038547,149.84371,2.228315,42.173244,7.250505,3.0,89.517571,...,1.712255,11.924652,1.66232,3.0,18.86644,6.634412,3.0,12,12,12
02a3099f15638eb904d611d076f62a7d27ecf085357b1bf3590dfc3c23adc1a6,90.682299,17.108042,6.204545,94.441913,25.184232,2.468929,61.432064,4.680531,6.0,82.21187,...,2.605356,51.369852,4.365846,6.5,55.635774,3.857188,3.25,12,12,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fec85eccc7c93dfc64e02b8607289cb714b5378a7b2d942d9c9821edbaa5e44d,250.071251,46.162957,5.295455,328.344557,88.381557,3.527101,131.379261,23.460259,4.5,141.568838,...,2.708013,143.735736,23.446856,4.0,415.33389,68.335401,2.5,12,12,12
fee99c844e544e3e07f8fe6c0b275a1c76f32c766365cf9c95b80dcc1bc1e316,761.968361,97.428056,5.840909,658.163245,35.445007,2.787024,472.167681,81.470155,5.0,662.820132,...,2.874918,516.783389,91.463531,7.5,1257.536578,21.368516,5.25,12,12,12
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,104.085906,7.447117,8.840909,114.340009,7.557776,2.232911,80.493482,5.428907,9.0,69.681356,...,2.167249,96.320114,6.153,9.5,127.121305,7.621908,2.5,12,12,12


In [45]:
cols = [c for c in list(df.columns) if "distance_straight_line" in c]

df.loc[:, cols]

,distance_straight_line_base_all_mean,distance_straight_line_base_all_std,distance_straight_line_base_all_median,distance_straight_line_base_all_iqr,distance_straight_line_base_all_n,distance_straight_line_base_weekend_mean,distance_straight_line_base_weekend_std,distance_straight_line_base_weekend_median,distance_straight_line_base_weekend_iqr,distance_straight_line_base_weekend_n,...,distance_straight_line_base_sun_mean,distance_straight_line_base_sun_std,distance_straight_line_base_sun_median,distance_straight_line_base_sun_iqr,distance_straight_line_base_sun_n,distance_straight_line_base_mon_mean,distance_straight_line_base_mon_std,distance_straight_line_base_mon_median,distance_straight_line_base_mon_iqr,distance_straight_line_base_mon_n
uid,,,,,,,,,,,,,,,,,,,,,
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,72.203654,50.767251,55.30733,35.398744,88,77.6054,54.259797,59.591722,23.164985,25,...,75.209794,64.774342,58.616518,31.257776,12,67.765381,50.770741,52.195381,30.059935,12
014d302436233505e32eb98250e9fae52e0cd079269ed8e3949838f740aeedd3,70.34959,118.50602,25.833019,58.033164,88,86.600793,148.310466,22.77981,82.783808,25,...,67.007392,190.251535,7.229214,14.379478,12,85.729209,180.012988,31.020494,54.743476,12
0263a0771738b6f9c0e2bf6ab61b963da028a0ef0dfa6ffb7db574441a66789c,411.656516,673.466929,120.87848,358.035184,88,203.580672,328.498858,68.925967,183.808742,25,...,128.390347,122.859957,112.650797,173.591059,12,180.961768,175.100962,103.989883,89.869062,12
0289ccadab6bfc8e61b02a43b73a75c29af9e720b8b5acdd245d89dfccb5bf46,208.81581,653.038547,42.173244,89.517571,88,380.75551,1084.399375,37.388637,70.992906,25,...,687.419949,1524.749824,20.612643,143.725045,12,174.52741,434.405391,11.924652,18.86644,12
02a3099f15638eb904d611d076f62a7d27ecf085357b1bf3590dfc3c23adc1a6,90.682299,94.441913,61.432064,82.21187,88,117.269454,131.441339,51.626563,185.347081,25,...,72.920951,71.881978,46.27198,76.991583,12,68.244392,38.360968,51.369852,55.635774,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fec85eccc7c93dfc64e02b8607289cb714b5378a7b2d942d9c9821edbaa5e44d,250.071251,328.344557,131.379261,141.568838,88,206.836488,223.560445,120.959339,152.320262,25,...,232.287686,294.785725,116.487307,161.695876,12,499.600274,703.036563,143.735736,415.33389,12
fee99c844e544e3e07f8fe6c0b275a1c76f32c766365cf9c95b80dcc1bc1e316,761.968361,658.163245,472.167681,662.820132,88,715.940679,607.924514,435.475404,557.95947,25,...,850.380796,661.94858,428.402553,1258.025869,12,1131.700206,1027.200088,516.783389,1257.536578,12
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,104.085906,114.340009,80.493482,69.681356,88,80.243341,46.196545,78.261253,40.682298,25,...,89.552745,54.818507,74.176297,37.107178,12,114.005002,78.035842,96.320114,127.121305,12


In [34]:
all_data = combined_data.merge(df, left_index=True, right_index=True)
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms_mobility_stats/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}_stats.parquet"
all_data.to_parquet(join(save_out_path, file_path))

### Calculate Deltas between AR Mobility Metrics and Baseline Mobility Metrics Statistics

In [40]:
column_heads=['base_all', 'base_weekend', 'base_weekday', 'base_fri', 'base_sat', 'base_sun', 'base_mon']
mobility_metrics = ['distance_straight_line', 'max_distance_from_home', 'num_stops']
column_heads_weekdays = {'base_all': [0,1,2,3,4,5,6],
                         'base_weekend': [5,6],
                         'base_weekday': [0,1,2,3,4],
                         'base_fri': [4],
                         'base_sat': [5],
                         'base_sun': [6],
                         'base_mon': [0]}

series_list = []
for AR_date in AR_dates:
    weekday = AR_date.day_of_week
    data_dir = f"/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251110/"
    fn = f"veraset_{AR_date.strftime('%Y-%m-%d')}_PST_chunked.parquet"
    metrics = pd.read_parquet(data_dir + fn)
    metrics = metrics[metrics.uid.isin(all_data.index)].set_index("uid", drop=True)
    for column_head in column_heads:
        if weekday in column_heads_weekdays[column_head]:
            for mobility_metric in mobility_metrics:
                col = mobility_metric + '_' + column_head
                all_data[col+'_std'] = all_data[col+'_std'].clip(0.0001)
                all_data[col+'_iqr'] = all_data[col+'_iqr'].clip(0.0001)
                delta = (metrics[mobility_metric] - all_data[col+'_mean'])/all_data[col+'_std']
                series_list.append(pd.Series(delta, name=f"{AR_date.strftime('%Y-%m-%d')}_{col}_z", index=all_data.index))
                delta = (metrics[mobility_metric] - all_data[col+'_median'])/all_data[col+'_iqr']
                series_list.append(pd.Series(delta, name=f"{AR_date.strftime('%Y-%m-%d')}_{col}_modz", index=all_data.index))
        else:
            continue
df_deltas = pd.concat(series_list, axis=1, join="inner")
df_deltas

,2024-03-29_distance_straight_line_base_all_z,2024-03-29_distance_straight_line_base_all_modz,2024-03-29_max_distance_from_home_base_all_z,2024-03-29_max_distance_from_home_base_all_modz,2024-03-29_num_stops_base_all_z,2024-03-29_num_stops_base_all_modz,2024-03-29_distance_straight_line_base_weekday_z,2024-03-29_distance_straight_line_base_weekday_modz,2024-03-29_max_distance_from_home_base_weekday_z,2024-03-29_max_distance_from_home_base_weekday_modz,...,2024-04-01_max_distance_from_home_base_weekday_z,2024-04-01_max_distance_from_home_base_weekday_modz,2024-04-01_num_stops_base_weekday_z,2024-04-01_num_stops_base_weekday_modz,2024-04-01_distance_straight_line_base_mon_z,2024-04-01_distance_straight_line_base_mon_modz,2024-04-01_max_distance_from_home_base_mon_z,2024-04-01_max_distance_from_home_base_mon_modz,2024-04-01_num_stops_base_mon_z,2024-04-01_num_stops_base_mon_modz
uid,,,,,,,,,,,,,,,,,,,,,
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,-0.778835,-0.639655,-0.427823,-1.283068,-1.385322,-1.0,-0.753901,-0.550428,-0.862305,-1.242912,...,-0.404075,-0.188469,-0.283866,0.0,-0.16564,0.238202,-0.297782,-0.03813,-0.127942,0.0
014d302436233505e32eb98250e9fae52e0cd079269ed8e3949838f740aeedd3,0.325726,1.432234,1.943981,4.671699,0.145096,0.235294,0.42869,1.558432,2.074853,5.005726,...,-0.576902,-0.611675,0.475536,0.571429,-0.391386,-0.28763,-0.543035,-0.780576,0.021393,-0.222222
0263a0771738b6f9c0e2bf6ab61b963da028a0ef0dfa6ffb7db574441a66789c,1.062038,2.809851,2.389856,1.71255,-0.478594,-0.4,0.837792,2.715641,2.265417,1.816711,...,-0.720005,-0.070316,0.135486,0.0,-0.767184,-0.638292,-0.653286,-0.082961,0.734313,0.5
0289ccadab6bfc8e61b02a43b73a75c29af9e720b8b5acdd245d89dfccb5bf46,-0.304391,-0.358994,-0.310987,-0.19756,0.678254,0.5,-0.366948,-0.372257,-0.315269,-0.216659,...,-0.306258,-0.154388,-0.728108,-0.5,-0.355109,0.442135,-0.413899,0.476598,-0.438019,-0.333333
02a3099f15638eb904d611d076f62a7d27ecf085357b1bf3590dfc3c23adc1a6,-0.510552,-0.230712,-0.521388,-0.040164,-0.487882,-0.25,-0.511174,-0.321389,-0.477707,-0.131956,...,1.57125,7.317664,1.035412,0.857143,0.672232,0.766809,1.398304,8.666034,0.895591,0.769231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fec85eccc7c93dfc64e02b8607289cb714b5378a7b2d942d9c9821edbaa5e44d,-0.112812,0.576756,-0.176188,1.002985,0.76679,1.076923,-0.149811,0.782315,-0.167841,20.73461,...,4.948414,1549.43393,1.710261,2.285714,4.926158,9.195334,1.925601,7.815688,2.46183,3.2
fee99c844e544e3e07f8fe6c0b275a1c76f32c766365cf9c95b80dcc1bc1e316,0.650434,1.083088,1.416177,2.541014,-0.301723,0.0,0.601909,1.084882,1.508989,4.168975,...,-0.196392,0.651649,-0.654567,-0.2,0.734336,1.088817,-0.277921,0.000251,-0.898576,-0.666667
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,0.279866,0.797808,0.731105,1.697761,0.519094,0.333333,0.17186,0.561388,0.643962,1.161166,...,0.39237,0.857294,0.386331,0.333333,-0.428089,-0.123672,0.868956,0.616771,0.384512,0.2


In [44]:
cols = [c for c in list(df_deltas.columns) if "distance_straight_line" in c]

df_deltas.loc[:, cols]

,2024-03-29_distance_straight_line_base_all_z,2024-03-29_distance_straight_line_base_all_modz,2024-03-29_distance_straight_line_base_weekday_z,2024-03-29_distance_straight_line_base_weekday_modz,2024-03-29_distance_straight_line_base_fri_z,2024-03-29_distance_straight_line_base_fri_modz,2024-03-30_distance_straight_line_base_all_z,2024-03-30_distance_straight_line_base_all_modz,2024-03-30_distance_straight_line_base_weekend_z,2024-03-30_distance_straight_line_base_weekend_modz,...,2024-03-31_distance_straight_line_base_weekend_z,2024-03-31_distance_straight_line_base_weekend_modz,2024-03-31_distance_straight_line_base_sun_z,2024-03-31_distance_straight_line_base_sun_modz,2024-04-01_distance_straight_line_base_all_z,2024-04-01_distance_straight_line_base_all_modz,2024-04-01_distance_straight_line_base_weekday_z,2024-04-01_distance_straight_line_base_weekday_modz,2024-04-01_distance_straight_line_base_mon_z,2024-04-01_distance_straight_line_base_mon_modz
uid,,,,,,,,,,,,,,,,,,,,,
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,-0.778835,-0.639655,-0.753901,-0.550428,-1.168521,-0.781137,-0.384783,-0.074524,-0.459569,-0.298832,...,-0.761742,-1.006619,-0.601108,-0.714802,-0.253075,0.114365,-0.215801,0.131195,-0.16564,0.238202
014d302436233505e32eb98250e9fae52e0cd079269ed8e3949838f740aeedd3,0.325726,1.432234,0.42869,1.558432,0.13372,0.657907,0.143967,1.061074,0.00546,0.780717,...,-0.559906,-0.232159,-0.333488,-0.255113,-0.464744,-0.181937,-0.462725,-0.214532,-0.391386,-0.28763
0263a0771738b6f9c0e2bf6ab61b963da028a0ef0dfa6ffb7db574441a66789c,1.062038,2.809851,0.837792,2.715641,0.185775,0.779313,NaN,NaN,NaN,NaN,...,0.081404,0.878064,0.829655,0.677863,-0.542015,-0.207386,-0.592711,-0.235607,-0.767184,-0.638292
0289ccadab6bfc8e61b02a43b73a75c29af9e720b8b5acdd245d89dfccb5bf46,-0.304391,-0.358994,-0.366948,-0.372257,-0.366368,-0.419036,-0.130952,0.906253,-0.237419,1.210123,...,-0.351005,-0.524884,-0.450759,-0.142543,-0.288727,-0.244724,-0.338195,-0.259801,-0.355109,0.442135
02a3099f15638eb904d611d076f62a7d27ecf085357b1bf3590dfc3c23adc1a6,-0.510552,-0.230712,-0.511174,-0.321389,-0.701991,-0.896815,-0.63562,-0.374385,-0.658973,-0.113157,...,-0.497443,0.001393,-0.292648,0.072902,0.035467,0.396534,0.188635,0.502291,0.672232,0.766809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fec85eccc7c93dfc64e02b8607289cb714b5378a7b2d942d9c9821edbaa5e44d,-0.112812,0.576756,-0.149811,0.782315,-0.065868,0.74359,2.661556,7.011425,4.102435,6.584937,...,0.247799,0.927488,0.101589,0.901367,11.307628,27.064502,10.215257,36.903222,4.926158,9.195334
fee99c844e544e3e07f8fe6c0b275a1c76f32c766365cf9c95b80dcc1bc1e316,0.650434,1.083088,0.601909,1.084882,0.558259,0.792314,2.470673,2.890538,2.750562,3.499536,...,6.395147,7.470492,5.670118,3.318934,1.707847,2.133072,1.624047,2.137822,0.734336,1.088817
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,0.279866,0.797808,0.17186,0.561388,0.244927,0.597826,0.010078,0.355113,0.541055,0.663113,...,-0.997473,-1.083954,-1.010411,-1.078303,-0.205415,0.001511,-0.251237,-0.060574,-0.428089,-0.123672


In [41]:
all_data = all_data.merge(df_deltas, left_index=True, right_index=True)
save_out_path = "/glade/campaign/cisl/aiml/cat_flood/mrms/home_locations_mrms_mobility_stats_deltas/"
date_str = datetime.today().strftime("%Y%m%d_%H%M")
file_path = f"veraset_users_mrms_{date_str}_stats_deltas.parquet"
all_data.to_parquet(join(save_out_path, file_path))
all_data

,duration,lat,lng,zipcode,row_group,county,precip_max,precip_sum,streamflow_max,streamflow_sum,...,2024-04-01_max_distance_from_home_base_weekday_z,2024-04-01_max_distance_from_home_base_weekday_modz,2024-04-01_num_stops_base_weekday_z,2024-04-01_num_stops_base_weekday_modz,2024-04-01_distance_straight_line_base_mon_z,2024-04-01_distance_straight_line_base_mon_modz,2024-04-01_max_distance_from_home_base_mon_z,2024-04-01_max_distance_from_home_base_mon_modz,2024-04-01_num_stops_base_mon_z,2024-04-01_num_stops_base_mon_modz
uid,,,,,,,,,,,,,,,,,,,,,
0139c27600dfb71e0ac78705698841e6610eee05428e45e62fbda5eed04fdae4,35 days 06:42:43,34.005120,-117.905949,91748,359,LA,"[0.0, 0.5, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.200...","[0.0, 0.5, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.9, ...","[0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.1, ...","[0.0, 0.0, 0.0, 0.0, 0.6, 0.0, 0.0, 0.0, 0.7, ...",...,-0.404075,-0.188469,-0.283866,0.0,-0.16564,0.238202,-0.297782,-0.03813,-0.127942,0.0
014d302436233505e32eb98250e9fae52e0cd079269ed8e3949838f740aeedd3,78 days 17:15:22,34.082222,-117.886791,91723,381,LA,"[0.0, 0.30000000000000004, 0.0, 0.300000000000...","[0.0, 0.30000000000000004, 0.0, 0.300000000000...","[0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.2, ...","[0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 1.3, ...",...,-0.576902,-0.611675,0.475536,0.571429,-0.391386,-0.28763,-0.543035,-0.780576,0.021393,-0.222222
0263a0771738b6f9c0e2bf6ab61b963da028a0ef0dfa6ffb7db574441a66789c,25 days 08:05:10,34.279326,-119.241598,93003,700,VE,"[0.0, 0.0, 0.0, 0.8, 0.0, 0.0, 0.0, 0.0, 6.2, ...","[0.0, 0.0, 0.0, 0.8, 0.0, 0.0, 0.0, 0.0, 17.0,...","[0.0, 0.0, 0.0, 0.1, 0.1, 0.0, 0.0, 0.0, 0.9, ...","[0.0, 0.0, 0.0, 0.1, 0.1, 0.0, 0.0, 0.0, 3.800...",...,-0.720005,-0.070316,0.135486,0.0,-0.767184,-0.638292,-0.653286,-0.082961,0.734313,0.5
0289ccadab6bfc8e61b02a43b73a75c29af9e720b8b5acdd245d89dfccb5bf46,83 days 02:27:18,34.229312,-119.184467,93036,744,VE,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.9, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 14.70...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.600...",...,-0.306258,-0.154388,-0.728108,-0.5,-0.355109,0.442135,-0.413899,0.476598,-0.438019,-0.333333
02a3099f15638eb904d611d076f62a7d27ecf085357b1bf3590dfc3c23adc1a6,47 days 03:59:04,34.198041,-118.532080,91335,773,LA,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.7, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 14.50...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.2, ...",...,1.57125,7.317664,1.035412,0.857143,0.672232,0.766809,1.398304,8.666034,0.895591,0.769231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fec85eccc7c93dfc64e02b8607289cb714b5378a7b2d942d9c9821edbaa5e44d,40 days 06:06:38,34.064190,-118.113987,91755,74660,LA,"[0.0, 4.800000000000001, 0.0, 0.0, 1.0, 0.0, 0...","[0.0, 5.1000000000000005, 0.0, 0.0, 1.0, 0.0, ...","[0.0, 0.3, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.2, ...","[0.0, 1.1, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.9, ...",...,4.948414,1549.43393,1.710261,2.285714,4.926158,9.195334,1.925601,7.815688,2.46183,3.2
fee99c844e544e3e07f8fe6c0b275a1c76f32c766365cf9c95b80dcc1bc1e316,53 days 04:02:21,34.969292,-120.425569,93454,74698,SB,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11.5,...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.3, ...",...,-0.196392,0.651649,-0.654567,-0.2,0.734336,1.088817,-0.277921,0.000251,-0.898576,-0.666667
ff39619779f97fb2dbce197c4d3d72d344c93b86f4a87b108ed717859a5ae244,70 days 00:04:38,33.828681,-118.070649,90716,74790,LA,"[0.0, 0.7000000000000001, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.7000000000000001, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.799...",...,0.39237,0.857294,0.386331,0.333333,-0.428089,-0.123672,0.868956,0.616771,0.384512,0.2
